In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

Matplotlib is building the font cache; this may take a moment.


In [ ]:

# 1. Load entire datasets without skipping records



train_df = pd.read_csv('train.csv/train.csv')
test_df = pd.read_csv('test.csv/test.csv')

In [ ]:
train_df

In [ ]:
train_df.isnull().sum()

In [ ]:
test_df

In [ ]:
test_df.isnull().sum()

In [ ]:
# Separate features and target (Target has no missing records in this dataset)
X_train = train_df.drop(columns=['Purchase', 'User_ID', 'Product_ID'])
y_train = train_df['Purchase']
X_test = test_df.drop(columns=['User_ID', 'Product_ID'])

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
# 2. Identify Column Types for Pipeline
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()

In [ ]:
cat_cols

In [ ]:
num_cols

In [ ]:
# 3. Build Transform Transformers (Handling NaNs dynamically)
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
num_transformer

In [ ]:
cat_transformer

In [ ]:

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

In [ ]:
preprocessor

In [ ]:
# 4. Create Full End-to-End Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))
])

In [ ]:
pipeline

In [ ]:
# 5. Fit on Train and Predict on Test
pipeline.fit(X_train, y_train)
test_predictions = pipeline.predict(X_test)

In [ ]:
# Output results mapped back to the entire test set
test_df['Predicted_Purchase'] = test_predictions
print(test_df[['User_ID', 'Product_ID', 'Predicted_Purchase']])

In [ ]:
# Label datasets for comparison
train_df['Dataset'] = 'Train'
test_df['Dataset'] = 'Test'
combined = pd.concat([train_df, test_df], axis=0)

# Set up visualization grid
fig, axes = plt.subplots(1, 3, figsize=(25, 10))

# Plot 1: Univariate Count Plot (Categorical Comparison)
sns.countplot(data=combined, x='Age', hue='Dataset', ax=axes[0], order=sorted(combined['Age'].unique()))
axes[0].set_title('Age Distribution: Train vs Test',fontsize=20)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Bivariate Distribution (Categorical vs Numerical)
# Note: 'Purchase' is only available in the train dataset
sns.boxplot(data=train_df, x='City_Category', y='Purchase', ax=axes[1])
axes[1].set_title('Train Only: Purchase by City Category', fontsize=20)

# Plot 3: Multivariate / Feature Proportions Overlap
sns.countplot(data=combined, x='City_Category', hue='Dataset', ax=axes[2])
axes[2].set_title('City Category Distribution: Train vs Test',fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Data
train_df = pd.read_csv('train.csv/train.csv').dropna(subset=['Purchase'])
test_df = pd.read_csv('test.csv/test.csv')

# 2. Extract EDA Summary Features
eda_summary = train_df.describe(include='all')

# 3. Process & Train
features = ['Gender', 'Age', 'City_Category']
X_train = pd.get_dummies(train_df[features], drop_first=True)
y_train = train_df['Purchase']
X_test = pd.get_dummies(test_df[features], drop_first=True).reindex(columns=X_train.columns, fill_value=0)

model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 4. Generate Predictions & Error Metrics
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

metrics_data = {
    'Metric': ['MAE', 'MSE', 'RMSE', 'R2 Score'],
    'Value': [
        mean_absolute_error(y_train, train_preds),
        mean_squared_error(y_train, train_preds),
        np.sqrt(mean_squared_error(y_train, train_preds)),
        r2_score(y_train, train_preds)
    ]
}
metrics_df = pd.DataFrame(metrics_data)

# 5. Save everything into Excel sheets
with pd.ExcelWriter('ml_complete_pipeline_results.xlsx') as writer:
    eda_summary.to_excel(writer, sheet_name='EDA_Summary')
    
    actual_pred_df = pd.DataFrame({'Actual_Purchase': y_train, 'Predicted_Purchase': train_preds})
    actual_pred_df.to_excel(writer, sheet_name='Actuals_and_Predicted', index=False)
    
    metrics_df.to_excel(writer, sheet_name='Error_Metrics', index=False)
    
    test_df['Predicted_Purchase'] = test_preds
    test_df[['User_ID', 'Product_ID', 'Predicted_Purchase']].to_excel(writer, sheet_name='Output_Predictions', index=False)

In [ ]:
import os
import warnings

# Bypasses verification for packages like requests and urllib
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["PYTHONHTTPSVERIFY"] = "0"

# Suppress the resulting insecure connection warnings
warnings.filterwarnings("ignore")

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, KFold
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Ingest Data & Setup EDA Summary
train_df = pd.read_csv('train.csv/train.csv')
test_df = pd.read_csv('test.csv/test.csv')
eda_summary = train_df.describe(include='all')

# 2. Optimized Pipeline Transformations
global_mean = train_df['Purchase'].mean()
for col in ['Product_ID', 'User_ID']:
    means = train_df.groupby(col)['Purchase'].mean()
    train_df[f'{col}_Enc'] = train_df[col].map(means).fillna(global_mean)
    test_df[f'{col}_Enc'] = test_df[col].map(means).fillna(global_mean)

for df in [train_df, test_df]:
    df['Age_Numeric'] = df['Age'].str.extract(r'(\d+)').astype(float).fillna(-1)
    df['Stay_Years'] = df['Stay_In_Current_City_Years'].str.replace('+', '', regex=False).astype(float).fillna(0)
    df['Product_Category_2'] = df['Product_Category_2'].fillna(-1)

features = ['Occupation', 'Marital_Status', 'Product_Category_1', 'Product_Category_2', 'Product_ID_Enc', 'User_ID_Enc', 'Age_Numeric', 'Stay_Years']
X, y = train_df[features], train_df['Purchase']

# 3. Multi-Model Search Orchestration
model_spaces = {
    'lgb': (LGBMRegressor(random_state=42, verbose=-1), {'n_estimators': [100, 150, 250], 'learning_rate': [0.001, 0.001, 0.001], 'num_leaves': [31, 67, 127]}),
    'xgb': (XGBRegressor(random_state=42, n_jobs=-1), {'n_estimators': [100, 150, 250], 'learning_rate': [0.001, 0.001, 0.001], 'max_depth': [6,7, 8]}),
    'cat': (CatBoostRegressor(verbose=0, random_state=42), {'iterations': [100, 150, 250], 'learning_rate': [0.001, 0.001, 0.001], 'depth': [6,7, 8]})
}

train_preds, test_preds, search_logs = np.zeros(len(train_df)), np.zeros(len(test_df)), []

for name, (model, space) in model_spaces.items():
    rs = RandomizedSearchCV(model, space, n_iter=5, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='r2', n_jobs=-1, random_state=42)
    rs.fit(X, y)
    best_estimator = rs.best_estimator_
    train_preds += best_estimator.predict(X) / len(model_spaces)
    test_preds += best_estimator.predict(test_df[features]) / len(model_spaces)
    search_logs.append({'Model': name, 'Optimal_Params': str(rs.best_params_), 'Best_CV_R2': f"{rs.best_score_*100:.2f}%"})

# 4. Assessment Matrix
metrics_df = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R2 Score (Accuracy)'],
    'Value': [mean_absolute_error(y, train_preds), mean_squared_error(y, train_preds), np.sqrt(mean_squared_error(y, train_preds)), r2_score(y, train_preds)]
})

# 5. Export to Multi-Sheet Workbook
with pd.ExcelWriter('model_final_outputs.xlsx') as writer:
    eda_summary.to_excel(writer, sheet_name='EDA_Summary')
    pd.DataFrame({'Actual_Purchase': y, 'Predicted_Purchase': train_preds}).to_excel(writer, sheet_name='Actuals_and_Predicted', index=False)
    metrics_df.to_excel(writer, sheet_name='Error_Metrics', index=False)
    pd.DataFrame(search_logs).to_excel(writer, sheet_name='Search_Summary', index=False)
    test_df['Predicted_Purchase'] = test_preds
    test_df[['User_ID', 'Product_ID', 'Predicted_Purchase']].to_excel(writer, sheet_name='Output_Sheet', index=False)

print("RandomizedSearchCV optimization completed successfully. Multi-sheet spreadsheet compiled.")

KeyboardInterrupt: 